# Syllabus to Song — free ACE-Step GPU

Uses the same official ACE-Step 1.5 REST technology found in your uploaded Android app.
Select **Runtime → Change runtime type → T4 GPU**, then run cells 1–4.
You sign into your own Google Colab account; no Google AI API or paid music service is used.
Free GPUs have availability, usage and session limits. Do not upgrade or enable billing for this prototype.

First setup downloads several GB of model files. Keep this notebook running while using the website.
The notebook is syntax-checked, but **has not been executed on a Colab GPU by this project**.
Only successful cell 3 output and your listening review can establish actual singing quality.
Official source pinned to `ca1e85fe9430179831e6bc6be790c332190a3866`.


In [ ]:
# 1. Install the pinned official model runtime (no training, no paid API).
import os, sys, subprocess, pathlib, time, json, secrets, urllib.parse
subprocess.run(['nvidia-smi'], check=True)
ROOT = pathlib.Path('/content/ACE-Step-1.5')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/ace-step/ACE-Step-1.5.git', str(ROOT)])
subprocess.check_call(['git', 'checkout', 'ca1e85fe9430179831e6bc6be790c332190a3866'], cwd=ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
subprocess.check_call(['uv', 'sync', '--frozen', '--python', '3.12'], cwd=ROOT)
print('Dependencies installed. Continue to cell 2.')


In [ ]:
# 2. Start the private official API. Load the DiT only to reduce GPU memory use.
import requests
# T4 cannot use native BF16. The pinned loader suggests a precision flag but
# does not implement it: apply an explicit, version-checked FP32 override.
loader = ROOT / 'acestep/core/generation/handler/init_service_orchestrator.py'
source = loader.read_text()
marker = '# Syllabus to Song: explicit T4 precision'
if marker not in source:
    anchor = '            self.quantization = normalized_quantization'
    assert source.count(anchor) == 1, 'Unexpected loader version; stop and inspect.'
    patch = ('            # Syllabus to Song: explicit T4 precision\n'
             '            if os.environ.get("SYLLABUS_T4_FP32") == "1":\n'
             '                self.dtype = torch.float32\n')
    source = source.replace(anchor, patch + anchor)
    compile(source, str(loader), 'exec')
    loader.write_text(source)
previous = globals().get('engine')
if previous is not None and previous.poll() is None:
    previous.terminate()
    try:
        previous.wait(timeout=30)
    except subprocess.TimeoutExpired:
        previous.kill(); previous.wait()
key_file = pathlib.Path('/content/.syllabus-song-key')
API_KEY = key_file.read_text().strip() if key_file.exists() else secrets.token_urlsafe(32)
key_file.write_text(API_KEY); key_file.chmod(0o600)
env = dict(os.environ, ACESTEP_API_KEY=API_KEY, ACESTEP_DEVICE='cuda',
           ACESTEP_CONFIG_PATH='acestep-v15-turbo', ACESTEP_INIT_LLM='false',
           ACESTEP_OFFLOAD_TO_CPU='true', ACESTEP_USE_FLASH_ATTENTION='false',
           SYLLABUS_T4_FP32='1', PYTHONUNBUFFERED='1', MPLBACKEND='Agg')
LOG = pathlib.Path('/content/ace-server.log')
engine = subprocess.Popen([str(ROOT / '.venv/bin/acestep-api'), '--host', '127.0.0.1', '--port', '8001'],
                          cwd=ROOT, env=env, stdout=LOG.open('w'), stderr=subprocess.STDOUT)
headers = {'Authorization': 'Bearer ' + API_KEY}
for attempt in range(600):
    if engine.poll() is not None:
        print(LOG.read_text(errors='replace').replace(API_KEY, '[REDACTED]')[-8000:])
        raise RuntimeError('Model startup stopped. Review the redacted log above.')
    try:
        r = requests.post('http://127.0.0.1:8001/query_result', json={'task_id_list': []}, headers=headers, timeout=3)
        if r.ok and r.json().get('code') == 200:
            print('Authenticated API ready. Run cell 3 to test actual song generation.'); break
    except requests.RequestException:
        pass
    if attempt % 15 == 0:
        print('Loading model / downloading free weights…', attempt * 2, 'seconds')
    time.sleep(2)
else:
    raise TimeoutError('Startup exceeded 20 minutes. Inspect /content/ace-server.log without sharing the key.')


In [ ]:
# 3. Real end-to-end songs for BOTH Calculus and BEEE; listen before claiming success.
# Choose 'hi' for Hindi and rerun this cell to check Hindi singing too.
LANGUAGE = 'en'
assert LANGUAGE in ('en', 'hi')
from IPython.display import Audio, display
FEATURED = {'calc-01': {'en': "[Verse]\nContinuous from a to b,\nDifferentiable in between.\nIf both ends share one height,\nRolle's theorem brings insight.\n\n[Chorus]\nSome c lies inside,\nWhere the tangent's flat, no rise.\nf prime c is zero,\nCheck the rules and let it show.\n\n[Chorus]\nEqual ends, conditions true,\nZero slope comes into view!", 'hi': '[Verse]\nए से बी तक सतत कहानी,\nबीच में अवकलनीय निशानी।\nदोनों सिरों पर मान समान,\nरोल प्रमेय का रखो ध्यान।\n\n[Chorus]\nसमझो, गाओ, याद बसाओ,\nबीच का कोई सी बतलाओ।\nसी पर अवकलज शून्य मिलेगा,\nशर्त सही तो नियम चलेगा।\n\n[Chorus]\nदोनों सिरों पर मान समान,\nशर्तों का भी रखो ध्यान!'}, 'beee-01': {'en': "[Verse]\nV equals I times R,\nOhm's law takes us far.\nInductor voltage: L times current's rate,\nCapacitor current: C times voltage's rate.\n\n[Chorus]\nR turns energy into heat,\nL and C store to the beat.\nL stores in a magnetic field,\nC stores in an electric field.\n\n[Chorus]\nV equals I times R,\nLearn what all the symbols are!", 'hi': '[Verse]\nवी बराबर आई गुणा आर,\nओम का नियम रखो याद यार।\nएल में धारा की बदलती दर,\nएल से गुणा तो वोल्टेज पर।\n\n[Verse]\nसी में वोल्टेज की बदलती दर,\nसी से गुणा तो धारा पर।\n\n[Chorus]\nसमझो, गाओ, याद बसाओ,\nआर में ऊर्जा ऊष्मा पाओ।\nएल का क्षेत्र चुंबकीय कहलाए,\nसी विद्युत क्षेत्र में ऊर्जा बचाए।\n\n[Chorus]\nवी बराबर आई गुणा आर,\nअर्थ समझो हर सूत्र का यार!'}}
for topic in ('calc-01', 'beee-01'):
    payload = dict(prompt='Original educational pop song, clear sung vocals, melody, bass, drums, catchy chorus',
                   lyrics=FEATURED[topic][LANGUAGE], vocal_language=LANGUAGE,
                   bpm=100, audio_duration=30, task_type='text2music', audio_format='wav',
                   batch_size=1, inference_steps=8, thinking=False,
                   use_cot_caption=False, use_cot_language=False, key_scale='C Major', time_signature='4')
    r = requests.post('http://127.0.0.1:8001/release_task', json=payload, headers=headers, timeout=30)
    r.raise_for_status()
    task = r.json()['data']['task_id']
    for attempt in range(600):
        r = requests.post('http://127.0.0.1:8001/query_result', json={'task_id_list': [task]}, headers=headers, timeout=30)
        r.raise_for_status()
        rows = r.json()['data']
        if rows and rows[0]['status'] == 2:
            print(LOG.read_text(errors='replace').replace(API_KEY, '[REDACTED]')[-8000:])
            raise RuntimeError('The GPU could not generate the song; inspect the log above.')
        if rows and rows[0]['status'] == 1:
            result = json.loads(rows[0]['result'])[0]
            p = urllib.parse.urlparse(result['file'])
            assert p.path == '/v1/audio'
            audio = requests.get('http://127.0.0.1:8001' + p.path + '?' + p.query, headers=headers, timeout=60)
            audio.raise_for_status()
            output = pathlib.Path('/content') / (topic + '-' + LANGUAGE + '.wav')
            output.write_bytes(audio.content)
            print(topic, LANGUAGE, '— actual model output; listen and check academic meaning.')
            display(Audio(audio.content))
            break
        if attempt % 15 == 0:
            print(topic, '— generating vocals and instruments…')
        time.sleep(2)
    else:
        raise TimeoutError('Generation exceeded 20 minutes. Keep the notebook open and inspect the model log.')


In [ ]:
# 4. Connect this running GPU to the WEBSITE, including from your phone.
# Creates a private random API key protected tunnel; keep the connection file secret.
import re, urllib.request
from google.colab import files
r = requests.post('http://127.0.0.1:8001/query_result', json={'task_id_list': []}, headers=headers, timeout=15)
r.raise_for_status()
assert r.json().get('code') == 200
previous_tunnel = globals().get('tunnel')
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill(); previous_tunnel.wait()
binary = pathlib.Path('/content/cloudflared')
if not binary.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', str(binary))
    binary.chmod(0o755)
tunnel_log = pathlib.Path('/content/singing-tunnel.log')
tunnel = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8001'],
                          stdout=tunnel_log.open('w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log.read_text(errors='replace'))
    if match:
        url = match.group(0); break
    if tunnel.poll() is not None:
        raise RuntimeError('The tunnel stopped. Inspect /content/singing-tunnel.log.')
    time.sleep(1)
if not url:
    raise RuntimeError('Tunnel startup failed; inspect /content/singing-tunnel.log.')
config = pathlib.Path('/content/song-service.env')
config.write_text('LOCAL_MUSIC_URL=' + url + '\nLOCAL_MUSIC_API_KEY=' + API_KEY + '\n')
config.chmod(0o600)
print('Download this file, sign into the website as admin, open Connections, select it and click Connect and test GPU.')
print('Service host (allow this in cloud network settings):', urllib.parse.urlparse(url).hostname)
print('Keep this notebook running. Do not publish the connection file or key.')
files.download(str(config))
